# Bert - No Finetuning

In [4]:
# load packages
import pandas as pd
import torch
from transformers import BertTokenizer, BertModel, AutoTokenizer, AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import sys
import os
from tqdm.auto import tqdm
import time
import numpy as np


/home/s215005/traffic_project/Thesis-traffic-safety/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# load data
EXCEL_PATH = "data/....xlsx"
#df = pd.read_excel(EXCEL_PATH)  # For example, contains ["SUMMARY", "LABEL"]
COLUMN_RENAME_MAP = {
    "UHELDSDATO": "accident_date",
    "UHELDSART": "report_category",
    "KODE_UHELDSSITUATION": "encoded_accident_situation",
    "UHELDSSITUATION": "accident_situation",
    "UHELDSTEKST": "police_narrative",
    "AAR": "year",
}

def extract_and_combine(folder_path):
    all_dfs = []

    for file in os.listdir(folder_path):
        if file.endswith(".xlsx"):
            file_path = os.path.join(folder_path, file)

            # Load file
            xls = pd.ExcelFile(file_path)

            if len(xls.sheet_names) != 1:
                raise ValueError(f"{file} has more than one sheet")

            df = xls.parse(xls.sheet_names[0], header=2)

            # Rename columns
            df = df.rename(columns=COLUMN_RENAME_MAP)

            # Keep only needed columns
            df = df[list(COLUMN_RENAME_MAP.values())]

            # Optional: track source file
            df["source_file"] = file

            all_dfs.append(df)

    # Combine everything
    combined_df = pd.concat(all_dfs, ignore_index=True)
    return combined_df

df = extract_and_combine("data/data udtræk")

print("Number of imported entries")
print(len(df))


/home/s215005/traffic_project/Thesis-traffic-safety/.venv/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/home/s215005/traffic_project/Thesis-traffic-safety/.venv/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/home/s215005/traffic_project/Thesis-traffic-safety/.venv/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/home/s215005/traffic_project/Thesis-traffic-safety/.venv/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  w

Number of imported entries
898677


In [6]:
#transform data, to only get high quaility text data and labels for training

df["encoded_accident_situation"] = pd.to_numeric(
    df["encoded_accident_situation"], errors="coerce"
)

df["main_situation_class"] = (df["encoded_accident_situation"] // 100).astype("Int64")

labels = df["main_situation_class"].tolist() # Labels are 0, 1, 2, 4, 5, 6, 9
texts = df["police_narrative"].astype(str).tolist()

df["has_narrative"] = df["police_narrative"].notna()
df_full = df.copy()

df_text = df_full.loc[df_full["has_narrative"]].copy()
df_text["n_words"] = df_text["police_narrative"].str.split().str.len()
df = df_text[df_text["n_words"] >= 3].copy()

print(len(df_text))

print("Number of samples per label:")
print(df["main_situation_class"].value_counts().sort_index())


893607
Number of samples per label:
main_situation_class
0    205604
1    160923
2     57458
3     76071
4     59439
5     71973
6     87567
7     95849
8     49886
9     27500
Name: count, dtype: Int64


In [7]:
# Load pre-trained BERT (no fine-tuning)

#english model
print("Loading pre-trained BERT model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
print("Tokenizer loaded.")
model = AutoModel.from_pretrained("bert-base-uncased")
print("Model loaded.")

#danish model
#tokenizer = BertTokenizer.from_pretrained("Maltehb/danish-bert-botxo")
#model = BertModel.from_pretrained("Maltehb/danish-bert-botxo")

model.eval()  # Evaluation mode, no parameter updates
print("BERT model set to evaluation mode.")


Loading pre-trained BERT model and tokenizer...
Tokenizer loaded.
Model loaded.
BERT model set to evaluation mode.


In [11]:
def get_bert_embeddings(texts, batch_size=16, max_len=128, save_every=50):
    chunks = []
    current_chunk = []
    start_time = time.time()

    with torch.no_grad():
        for batch_num, i in enumerate(
            tqdm(range(0, len(texts), batch_size), desc="Extracting BERT embeddings")
        ):
            batch_texts = texts[i:i+batch_size]
            inputs = tokenizer(
                batch_texts,
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=max_len
            )

            outputs = model(**inputs)
            cls_embeddings = outputs.last_hidden_state[:, 0, :]
            current_chunk.append(cls_embeddings.numpy())

            if (batch_num + 1) % save_every == 0:
                chunks.append(np.vstack(current_chunk))
                current_chunk = []

        if current_chunk:
            chunks.append(np.vstack(current_chunk))

    X = np.vstack(chunks)
    print(f"Embedding extraction took {time.time() - start_time:.2f} seconds")
    return X

In [12]:
texts=sorted(texts, key=lambda x: len(x))
X = get_bert_embeddings(texts, batch_size=256)

Extracting BERT embeddings: 100%|██████████| 3511/3511 [6:51:38<00:00,  7.03s/it]  


Embedding extraction took 24698.80 seconds


In [ ]:
X_256 = X

In [13]:

X_train, X_test, y_train, y_test = train_test_split(X, labels, test_size=0.2, random_state=42, stratify=labels)


In [14]:
# Use Logistic Regression for classification
clf = LogisticRegression(max_iter=2000)
clf.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,2000
,multi_class,'deprecated'


In [15]:
from sklearn.metrics import accuracy_score, classification_report, f1_score
import numpy as np

# Original evaluation on all labels
y_pred = clf.predict(X_test)
print("=== All labels (including 9) ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro-F1:", f1_score(y_test, y_pred, average="macro"))
print(classification_report(y_test, y_pred, digits=3))

# Remove samples with label 9
mask = np.array(y_test) != 9
y_test_filtered = np.array(y_test)[mask]
y_pred_filtered = np.array(y_pred)[mask]

print("\n=== After removing label 9 ===")
print("Accuracy:", accuracy_score(y_test_filtered, y_pred_filtered))
print("Macro-F1:", f1_score(y_test_filtered, y_pred_filtered, average="macro"))
print(classification_report(y_test_filtered, y_pred_filtered, digits=3))


=== All labels (including 9) ===
Accuracy: 0.23010971647304937
Macro-F1: 0.04924137732233198
              precision    recall  f1-score   support

           0      0.232     0.932     0.371     41410
           1      0.209     0.085     0.121     32487
           2      0.000     0.000     0.000     11576
           3      0.000     0.000     0.000     15311
           4      0.000     0.000     0.000     11948
           5      0.000     0.000     0.000     14469
           6      0.200     0.000     0.000     17597
           7      0.000     0.000     0.000     19346
           8      0.000     0.000     0.000     10056
           9      0.000     0.000     0.000      5536

    accuracy                          0.230    179736
   macro avg      0.064     0.102     0.049    179736
weighted avg      0.111     0.230     0.107    179736


=== After removing label 9 ===
Accuracy: 0.23742250287026406
Macro-F1: 0.0558756735755158
              precision    recall  f1-score   support

  

/home/s215005/traffic_project/Thesis-traffic-safety/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/s215005/traffic_project/Thesis-traffic-safety/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/s215005/traffic_project/Thesis-traffic-safety/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter

In [16]:
# Load pre-trained BERT (no fine-tuning)

#english model
#print("Loading pre-trained BERT model and tokenizer...")
#tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
#print("Tokenizer loaded.")
#model = AutoModel.from_pretrained("bert-base-uncased")
#print("Model loaded.")

#danish model
print("Loading pre-trained BERT model and tokenizer...")
tokenizer = BertTokenizer.from_pretrained("Maltehb/danish-bert-botxo")
print("Tokenizer loaded.")
model = BertModel.from_pretrained("Maltehb/danish-bert-botxo")
print("Model loaded.")

model.eval()  # Evaluation mode, no parameter updates
print("BERT model set to evaluation mode.")


Loading pre-trained BERT model and tokenizer...
Tokenizer loaded.
Model loaded.
BERT model set to evaluation mode.


In [17]:
texts=sorted(texts, key=lambda x: len(x))

x_danish = get_bert_embeddings(texts, batch_size=128)

Extracting BERT embeddings: 100%|██████████| 7021/7021 [4:07:39<00:00,  2.12s/it]  


Embedding extraction took 14859.42 seconds


In [18]:
X_train_danish, X_test_danish, y_train_danish, y_test_danish = train_test_split(x_danish, labels, test_size=0.2, random_state=42, stratify=labels)

In [19]:
# Use Logistic Regression for classification
clf_danish = LogisticRegression(max_iter=2000)
clf_danish.fit(X_train_danish, y_train_danish)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,2000
,multi_class,'deprecated'


In [20]:
from sklearn.metrics import accuracy_score, classification_report, f1_score
import numpy as np

# Original evaluation on all labels
y_pred_danish = clf_danish.predict(X_test_danish)
print("=== All labels (including 9) ===")
print("Accuracy:", accuracy_score(y_test_danish, y_pred_danish))
print("Macro-F1:", f1_score(y_test_danish, y_pred_danish, average="macro"))
print(classification_report(y_test_danish, y_pred_danish, digits=3))

# Remove samples with label 9
mask = np.array(y_test_danish) != 9
y_test_filtered = np.array(y_test_danish)[mask]
y_pred_filtered = np.array(y_pred_danish)[mask]

print("\n=== After removing label 9 ===")
print("Accuracy:", accuracy_score(y_test_filtered, y_pred_filtered))
print("Macro-F1:", f1_score(y_test_filtered, y_pred_filtered, average="macro"))
print(classification_report(y_test_filtered, y_pred_filtered, digits=3))


=== All labels (including 9) ===
Accuracy: 0.22999287844394
Macro-F1: 0.04490618704028578
              precision    recall  f1-score   support

           0      0.231     0.961     0.373     41410
           1      0.197     0.047     0.076     32487
           2      0.000     0.000     0.000     11576
           3      0.000     0.000     0.000     15311
           4      0.000     0.000     0.000     11948
           5      0.000     0.000     0.000     14469
           6      0.000     0.000     0.000     17597
           7      0.000     0.000     0.000     19346
           8      0.000     0.000     0.000     10056
           9      0.000     0.000     0.000      5536

    accuracy                          0.230    179736
   macro avg      0.043     0.101     0.045    179736
weighted avg      0.089     0.230     0.100    179736


=== After removing label 9 ===
Accuracy: 0.23730195177956373
Macro-F1: 0.05100117834876385
              precision    recall  f1-score   support

    

/home/s215005/traffic_project/Thesis-traffic-safety/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/s215005/traffic_project/Thesis-traffic-safety/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/s215005/traffic_project/Thesis-traffic-safety/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter